# Train Smart Helmet Detection (YOLO11)

Dataset: [Helmet Detection_YOLOv8 v3](https://universe.roboflow.com/learning-evidence/helmet-detection_yolov8) (CC BY 4.0) — 2 คลาส: `With Helmet`, `Without Helmet`

**ก่อนเริ่ม:** เมนู Runtime → Change runtime type → เลือก **T4 GPU** → Save แล้วกดรันทีละ cell จากบนลงล่าง

## 1. เช็ค GPU
ต้องเห็นคำว่า `Tesla T4` ถ้าขึ้น error ให้กลับไปเปิด GPU ก่อน

In [ ]:
!nvidia-smi

## 2. เชื่อม Google Drive
ผล train จะถูกเซฟลง Drive โดยตรง ถ้า Colab หลุดก็ไม่หาย

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/helmet-detection'

## 3. ติดตั้งไลบรารี

In [ ]:
!pip install -q ultralytics roboflow

## 4. ดาวน์โหลด dataset
ต้องใช้ Roboflow API key (ฟรี): ล็อกอิน roboflow.com → Settings → API Keys → คัดลอก **Private API Key** แล้ววางในช่องที่ขึ้นมา

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass('Roboflow API key: '))
dataset = rf.workspace('learning-evidence').project('helmet-detection_yolov8').version(3).download('yolov11')
DATA_YAML = f'{dataset.location}/data.yaml'
print(open(DATA_YAML).read())

## 5. Train
`yolo11n` เล็กและเร็ว (ประมาณ 30–60 นาทีบน T4) ถ้ามีเวลาเหลือ ลองเปลี่ยนเป็น `yolo11s.pt` จะแม่นขึ้น

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,          # หยุดก่อนถ้าไม่ดีขึ้น 15 epoch ติด
    project=SAVE_DIR,
    name='yolo11n',
    exist_ok=True,
)

## 6. ประเมินผลบน test set
ตัวเลขสำหรับใส่รายงาน: Precision, Recall, mAP50, mAP50-95 แยกตามคลาส

In [ ]:
best = YOLO(f'{SAVE_DIR}/yolo11n/weights/best.pt')
metrics = best.val(data=DATA_YAML, split='test')
print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'Precision : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')
for i, c in enumerate(metrics.box.ap_class_index):
    print(f'{best.names[int(c)]:15s} AP50 = {metrics.box.ap50[i]:.3f}')

## 7. กราฟสำหรับรายงาน

In [ ]:
import os
from IPython.display import Image, display
run = f'{SAVE_DIR}/yolo11n'
for f in ['results.png', 'confusion_matrix.png', 'BoxF1_curve.png', 'F1_curve.png', 'val_batch0_pred.jpg']:
    if os.path.exists(f'{run}/{f}'):
        print(f); display(Image(f'{run}/{f}', width=800))

## 8. ลองทำนายรูปจาก test set
รูปที่วาดกล่องแล้วจะเซฟไว้ที่ `SAVE_DIR/predict`

In [ ]:
import glob
imgs = sorted(glob.glob(f'{dataset.location}/test/images/*'))[:8]
best.predict(imgs, conf=0.5, save=True, project=SAVE_DIR, name='predict', exist_ok=True)
for p in sorted(glob.glob(f'{SAVE_DIR}/predict/*.jpg'))[:4]:
    display(Image(p, width=600))

## 9. ดาวน์โหลด `best.pt`
ไฟล์โมเดลนี้จะใช้ต่อใน Back-end (มีสำรองใน Google Drive ที่ `MyDrive/helmet-detection/yolo11n/weights/best.pt` ด้วย)

In [ ]:
from google.colab import files
files.download(f'{SAVE_DIR}/yolo11n/weights/best.pt')